# 02 · Análise dos Resultados

Comparação entre modelos, testes de significância, Ablation Study e
interpretabilidade — material para os capítulos de resultados e discussão da
dissertação.

> Este notebook **lê** os artefatos gravados por `make evaluate`. Não treina
> nem recalcula nada: reimplementar a avaliação aqui criaria uma segunda fonte
> de verdade, com risco de divergir do pipeline.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from config.paths import get_paths
from utils.files import read_json
from visualization.theme import apply_theme

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
apply_theme()
plt.rcParams["figure.dpi"] = 120

PATHS = get_paths()
EVALUATION = read_json(PATHS.reports.metrics / "evaluation.json")
PRIMARY = EVALUATION.get("primary_metric", "f1_macro")

## 1. Comparação entre modelos (H1, H4)

**Leitura:** a ordenação não é um ranking de superioridade comprovada.
Intervalos de confiança sobrepostos significam que a diferença não está
estabelecida, por maior que seja a diferença das médias.

In [ ]:
from visualization.evaluation_plots import plot_model_comparison

comparison = pl.read_csv(PATHS.reports.tables / "model_comparison.csv")
display(comparison.to_pandas())

plot_model_comparison(comparison, PRIMARY)
plt.show()

## 2. Validação cruzada

Média e desvio entre folds. A variação entre folds é, ela própria, informação:
um modelo com desvio alto é instável, mesmo com média boa.

In [ ]:
cv_path = PATHS.reports.metrics / "cross_validation.json"
if cv_path.is_file():
    cv = read_json(cv_path)
    records = [
        {
            "modelo": name,
            "média": result["mean"],
            "desvio": result["std"],
            "IC inferior": result.get("ci_lower"),
            "IC superior": result.get("ci_upper"),
        }
        for name, result in cv.items()
    ]
    display(pl.DataFrame(records).sort("média", descending=True).to_pandas())
else:
    print("Execute `make train` para gerar os resultados da validação cruzada.")

## 3. Desempenho por classe

A revocação de **Ideação Suicida** é a métrica de maior consequência clínica:
um falso negativo significa deixar de sinalizar alguém potencialmente em risco.

In [ ]:
from constants.labels import CLASS_DISPLAY_NAMES, CLASS_ORDER

models = EVALUATION.get("models", {})
best = max(models, key=lambda name: models[name]["metrics"].get(PRIMARY, 0.0))
print(f"Melhor modelo: {best}")

per_class = models[best]["per_class"]
display(
    pl.DataFrame(
        [{"classe": CLASS_DISPLAY_NAMES[name], **per_class.get(name, {})} for name in CLASS_ORDER]
    ).to_pandas()
)

In [ ]:
from visualization.evaluation_plots import plot_confusion_matrix, plot_reliability_curve

plot_confusion_matrix(models[best]["confusion_matrix"], title=f"Matriz de Confusão — {best}")
plt.show()

if models[best].get("calibration"):
    plot_reliability_curve(models[best]["calibration"], best)
    plt.show()
    print(models[best]["calibration"]["interpretation"])

## 4. Testes de significância

O p-valor responde *"a diferença é real?"*; o tamanho de efeito responde
*"a diferença importa?"*. Os dois são reportados lado a lado.

In [ ]:
statistics = EVALUATION.get("statistics", {})

if "friedman" in statistics:
    print("Friedman:", statistics["friedman"]["interpretation"], "\n")

if "pairwise" in statistics:
    display(
        pl.DataFrame(
            [
                {
                    "comparação": pair.replace("_vs_", " vs. "),
                    "p": entry["p_value"],
                    "p corrigido": entry["p_value_corrected"],
                    "significativo": entry["significant"],
                    "efeito": entry["effect_size"],
                }
                for pair, entry in statistics["pairwise"].items()
            ]
        ).to_pandas()
    )

## 5. Ablation Study (H2, H3, H4)

**Contribuição marginal** (leave-one-out) mede o que o grupo acrescenta além do
que os outros já capturam. **Apenas o grupo** (only-one) mede sua contribuição
absoluta. Grupos correlacionados têm marginal baixa sem serem inúteis.

In [ ]:
from visualization.interpretability_plots import plot_ablation

ablation_path = PATHS.reports.ablation / "ablation_summary.csv"
if ablation_path.is_file():
    ablation = pl.read_csv(ablation_path)
    display(ablation.to_pandas())
    plot_ablation(ablation, PRIMARY)
    plt.show()
else:
    print("Execute `make evaluate` para gerar o Ablation Study.")

## 6. Interpretabilidade

**Leitura correta:** SHAP e importância por permutação explicam o comportamento
do **modelo**, não a causalidade do fenômeno. Um valor alto significa que o
modelo se apoia naquela feature — não que ela cause risco.

In [ ]:
from visualization.interpretability_plots import (
    plot_feature_importance,
    plot_group_importance,
    plot_shap_summary,
)

for path in sorted(PATHS.reports.interpretability.glob("permutation_importance_*.csv")):
    plot_feature_importance(pl.read_csv(path))
    plt.show()

for path in sorted(PATHS.reports.interpretability.glob("group_importance_*.csv")):
    plot_group_importance(pl.read_csv(path))
    plt.show()

for path in sorted(PATHS.reports.interpretability.glob("shap_summary_*.csv")):
    plot_shap_summary(pl.read_csv(path))
    plt.show()

## 7. Avaliação por subgrupo

A métrica agregada esconde falhas concentradas: um F1-macro razoável é
compatível com desempenho próximo do aleatório em usuários de histórico curto.

In [ ]:
from visualization.evaluation_plots import plot_slice_performance

slices = models[best].get("slices", {})
if slices:
    figure = plot_slice_performance(slices, PRIMARY)
    if figure is not None:
        plt.show()

    for name, data in slices.items():
        if data.get("exceeds_threshold"):
            print(f"⚠️  Disparidade acima do limite na fatia '{name}': {data['gap']:.4f}")

## 8. Síntese das hipóteses

| Hipótese | Evidência | Onde verificar |
|---|---|---|
| **H1** Transformers > TF-IDF | Comparação `bertimbau` vs. `tfidf_logistic` | Seções 1 e 4 |
| **H2** Temporais/comportamentais ajudam | Contribuição marginal dos grupos | Seção 5 |
| **H3** Atributos de LLM ajudam | Contribuição marginal de `psychological` | Seção 5 |
| **H4** Híbrido generaliza melhor | `hybrid_xgboost` vs. demais, com teste | Seções 1 e 4 |
| **H5** Usuário > tweet | Comparação de granularidade | `reports/metrics/` |

**Ao redigir a discussão:** toda afirmação de superioridade precisa citar o
teste de significância *e* o tamanho de efeito, e vir acompanhada das limitações
declaradas no model card (viés de seleção do controle, rótulos sem validação
clínica, ausência de auditoria demográfica).